# Homework 2: Your First Regression Model Pipeline

## Learning goals

By the end of this assignment, you will be able to:

1. Frame a scientific prediction task as a supervised machine learning regression problem.
2. Separate predictors from a numeric target variable.
3. Split data into training and testing sets without leaking test information into model fitting.
4. Build a reproducible preprocessing and modeling workflow using a scikit-learn `Pipeline`.
5. Evaluate a regression model using MAE, RMSE, and R².
6. Interpret model performance and limitations in a scientific context.

## Dataset

We will use the **Palmer Penguins** dataset, which contains size measurements, clutch observations, and related field metadata for adult foraging Adélie, Chinstrap, and Gentoo penguins observed near Palmer Station in the Palmer Archipelago, Antarctica. The dataset was collected by Dr. Kristen Gorman and the Palmer Station Long Term Ecological Research Program.

**Regression task:** predict penguin body mass in grams from anatomical measurements and field metadata.

Dataset reference: Horst, Hill, and Gorman's `palmerpenguins` package; source data from Palmer Station LTER and Gorman (2020), Environmental Data Initiative.

## Submission instructions

Submit a completed notebook with:

- All code cells executed in order.
- Written answers in the reflection prompts.
- At least one model pipeline that runs successfully.
- One short interpretation of model performance in environmental or marine science terms.

## 0. Setup

Run the cell below. If a package is missing, install it in your environment before continuing.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.dummy import DummyRegressor
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42 # This makes sure everyone gets the same answer, it is 42 because that is the answer to life, the universe, and everything. 
pd.set_option("display.max_columns", 100)

## 1. Load the data

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/allisonhorst/palmerpenguins/main/inst/extdata/penguins.csv"

penguins = pd.read_csv(DATA_URL)
penguins.head()

### Exercise 1: Initial inspection

Use pandas to answer the following questions:

1. How many rows and columns are in the dataset?
2. Which columns are numeric? Which are categorical?
3. How many missing values are present in each column?
4. What is the range of the target variable, `body_mass_g`?

In [ ]:
# TODO: Inspect the shape of the dataframe.

In [ ]:
# TODO: Inspect column names, data types, and missing values.

In [ ]:
# TODO: Summarize the target variable body_mass_g.

Write 2–4 sentences interpreting the dataset. Consider whether this is a large or small dataset and what kinds of scientific uncertainty or sampling limitations might matter.

**Your answer:**

## 2. Define a regression problem

Our target variable is `body_mass_g`, a continuous numeric measurement. We will predict it from a mix of numeric and categorical predictors.

To avoid a circular prediction task, do **not** use `body_mass_g` as an input feature.

In [ ]:
target = "body_mass_g"

# We drop rows where the target is missing because supervised learning requires known labels.
data = penguins.dropna(subset=[target]).copy()

X = data.drop(columns=[target]) # Features
y = data[target] # Targert

print("Feature matrix shape:", X.shape)
print("Target vector shape:", y.shape)
X.head()

### Exercise 2: Scientific framing

In 3–5 sentences, describe this prediction task in scientific language. For example:

- What are we trying to estimate?
- What kinds of measurements might be informative predictors?

**Your answer:**

## 3. Split into training and test sets

The model will learn from the training set. The test set will be held aside until final evaluation.

This split helps us estimate how well the model may perform on new observations that were not used during model fitting.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

### Exercise 3: Why split the data?

In 2–3 sentences, explain why evaluating on the same data used for training can give an overly optimistic estimate of model performance.

**Your answer:**

## 4. Identify numeric and categorical predictors

A model pipeline lets us apply different preprocessing steps to different column types.

For this dataset:

- Numeric columns can be imputed and scaled.
- Categorical columns can be imputed and one-hot encoded.

In [ ]:
numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number"]).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

## 5. Build preprocessing steps

The `ColumnTransformer` below keeps preprocessing inside the model workflow. This is important because preprocessing should be learned from the training data only, then applied consistently to the test data.

In [ ]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

### Exercise 4: Explain the preprocessing

Briefly explain what each preprocessing step does:

1. `SimpleImputer(strategy="median")`
2. `StandardScaler()`
3. `SimpleImputer(strategy="most_frequent")`
4. `OneHotEncoder(handle_unknown="ignore")`

**Your answer:**

## 6. Build and train a baseline model

Before training a real model, we need a simple baseline. Here, the baseline always predicts the mean body mass from the training data.

A useful model should perform better than this simple benchmark.

In [ ]:
baseline_model = DummyRegressor(strategy="mean")
baseline_model.fit(X_train, y_train)

baseline_predictions = baseline_model.predict(X_test)

baseline_mae = mean_absolute_error(y_test, baseline_predictions)
baseline_rmse = root_mean_squared_error(y_test, baseline_predictions)
baseline_r2 = r2_score(y_test, baseline_predictions)

print(f"Baseline MAE:  {baseline_mae:.1f} g")
print(f"Baseline RMSE: {baseline_rmse:.1f} g")
print(f"Baseline R²:   {baseline_r2:.3f}")

## 7. Build and train your first regression pipeline

Now we combine preprocessing and a regression model into one pipeline.

This is the central pattern for many applied machine learning workflows:

```text
raw data → preprocessing → model → predictions
```

In [ ]:
linear_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression())
    ]
)

linear_pipeline.fit(X_train, y_train)

## 8. Evaluate the model on the test set

We will use three common regression metrics:

- **MAE**: mean absolute error, in grams. Easy to interpret.
- **RMSE**: root mean squared error, in grams. Penalizes larger errors more strongly.
- **R²**: proportion of variance explained relative to a mean-prediction baseline.

In [ ]:
y_pred = linear_pipeline.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Linear pipeline MAE:  {mae:.1f} g")
print(f"Linear pipeline RMSE: {rmse:.1f} g")
print(f"Linear pipeline R²:   {r2:.3f}")

### Exercise 5: Compare against the baseline

In 3–5 sentences, compare the linear regression pipeline to the baseline model. Address the following:

1. Did the model improve over the baseline?
2. Which metric is easiest for you to interpret scientifically?
3. Is the error small or large relative to the observed range of body mass?

**Your answer:**

## 9. Visualize predictions and residuals

Plots help reveal model behavior that summary metrics can hide.

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred, alpha=0.8)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], linestyle="--")
plt.xlabel("Observed body mass (g)")
plt.ylabel("Predicted body mass (g)")
plt.title("Observed vs. predicted penguin body mass")
plt.show()

In [ ]:
residuals = y_test - y_pred

plt.figure(figsize=(7, 4))
plt.scatter(y_pred, residuals, alpha=0.8)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted body mass (g)")
plt.ylabel("Residual: observed - predicted (g)")
plt.title("Residual plot")
plt.show()

### Exercise 6: Interpret the plots

In 4–6 sentences, describe what you see. Consider:

- Are predictions close to the 1:1 line?
- Are residuals centered around zero?
- Are there outliers or patterns?
- What might these patterns imply about model assumptions or missing predictors?

**Your answer:**

## 10. Use cross-validation on the training set

A single train/test split can be sensitive to which observations end up in each set. Cross-validation gives a more stable estimate of model performance by repeatedly fitting and evaluating the model on different training/validation folds.

Important: cross-validation below is run only on the training data. The test set remains reserved for final evaluation.

In [ ]:
cv_results = cross_validate(
    linear_pipeline,
    X_train,
    y_train,
    cv=5,
    scoring={
        "mae": "neg_mean_absolute_error",
        "rmse": "neg_root_mean_squared_error",
        "r2": "r2"
    },
    return_train_score=False
)

cv_summary = pd.DataFrame({
    "MAE_g": -cv_results["test_mae"],
    "RMSE_g": -cv_results["test_rmse"],
    "R2": cv_results["test_r2"]
})

cv_summary

In [ ]:
cv_summary.agg(["mean", "std"])

### Exercise 7: Cross-validation interpretation

In 2–4 sentences, compare the cross-validation results to the final test-set results. Are they broadly similar? What would it mean if they were very different?

**Your answer:**

## 11. Try a regularized regression model

Linear regression can work well, but regularized models are often useful because they discourage overly large coefficients. Here, we try Ridge regression, which is a common first regularized regression model.

In [ ]:
ridge_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", Ridge(alpha=1.0))
    ]
)

ridge_pipeline.fit(X_train, y_train)
ridge_pred = ridge_pipeline.predict(X_test)

ridge_mae = mean_absolute_error(y_test, ridge_pred)
ridge_rmse = root_mean_squared_error(y_test, ridge_pred)
ridge_r2 = r2_score(y_test, ridge_pred)

print(f"Ridge pipeline MAE:  {ridge_mae:.1f} g")
print(f"Ridge pipeline RMSE: {ridge_rmse:.1f} g")
print(f"Ridge pipeline R²:   {ridge_r2:.3f}")

### Exercise 8: Model comparison

Create a small table comparing the baseline, linear regression pipeline, and Ridge pipeline.

In [ ]:
# TODO: Create a dataframe with rows for baseline, linear regression, and Ridge.
# Include columns for MAE, RMSE, and R2.

In 3–5 sentences, state which model you would report and why. Your answer should include both predictive performance and scientific interpretability.

**Your answer:**

## 12. Modify the feature set

Choose one of the following experiments:

A. Remove categorical variables and use only anatomical measurements.  
B. Remove `species` and test whether the model still performs well.  
C. Use only field metadata such as island, species, sex, and year.  
D. Add your own features (e.g. `bill_legth_mm` + `flipper_length_mm`).  
  
Record what you changed and whether performance improved, declined, or stayed similar.

In [ ]:
# TODO: Modify the feature set and evaluate your experiment.

**Notes:**

## 13. Final reflection

Answer the following in a short paragraph or two:

1. What is one advantage of using a pipeline for applied scientific machine learning?
2. What is one risk or limitation of this model?
3. What additional data might improve predictions of penguin body mass?
4. How could a similar workflow be used in geology, environmental science, or marine science?

**Your final reflection:**

## Rubric

| Component | Points |
|---|---:|
| Data loading and inspection completed | 1 |
| Regression task clearly framed | 2 |
| Train/test split and feature/target separation correct | 1 |
| Preprocessing pipeline implemented correctly | 3 |
| Baseline and regression model evaluated with appropriate metrics | 3 |
| Plots generated and interpreted | 3 |
| Cross-validation completed and interpreted | 3 |
| Modified feature set | 6 |
| Final reflection connects ML workflow to Earth/environmental/marine science | 3 |
| **Total** | **25** |